# Module 2: Data Cleaning & Transformation


## Objective

The objective of this module is to clean, validate, and prepare the hospital dataset for KPI engineering and Tableau dashboard development.

The cleaning process improves data quality by removing duplicate records, handling missing values, validating numerical fields, and preparing a Tableau-ready dataset.
## Dataset

Input Dataset:
hospital_raw_data.csv

Source:
New York State Department of Health (SPARCS)

Output Dataset:
hospital_cleaned.csv

In [1]:
import pandas as pd
from pathlib import Path


In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [3]:
df = pd.read_csv(DATA_DIR / "hospital_raw_data.csv")

print(df.shape)
df.head()

(8000, 33)


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,70 or Older,OOS,M,Other Race,Unknown,...,Extreme,Extreme,Medical,Medicare,Private Health Insurance,NaN,NaN,N,121012.20,50785.63
1,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,0 to 17,114,F,Other Race,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,N,20798.65,9416.18
2,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,30 to 49,100,M,Black/African American,Spanish/Hispanic,...,Moderate,Moderate,Surgical,Medicaid,NaN,NaN,NaN,Y,124567.28,45270.48
3,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,70 or Older,103,F,Other Race,Unknown,...,Major,Major,Surgical,Medicare,NaN,NaN,NaN,N,325157.81,122411.36
4,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,0 to 17,100,M,White,Not Span/Hispanic,...,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,03500,N,21948.10,1071.12


In [4]:
clean_df = df.copy()

Module 2 Summary

- Removed 1 duplicate record.
- Filled missing ZIP codes with "Unknown".
- Filled missing diagnosis information with "Unknown".
- Filled missing procedure information with "No Procedure".
- Filled missing Payment Typology 2 with "Not Applicable".
- Filled missing Birth Weight with "Not Applicable".
- Dropped Payment Typology 3 due to 99.2% missing values and low relevance to the project.
- Validated numeric fields and found no invalid negative values.
- Final dataset contains 7,999 rows and 32 columns.

In [5]:
print("Duplicate rows before cleaning:", clean_df.duplicated().sum())

Duplicate rows before cleaning: 1


In [6]:
clean_df.drop_duplicates(inplace=True)


In [7]:
print("Duplicate rows after cleaning:", clean_df.duplicated().sum())
print(clean_df.shape)

Duplicate rows after cleaning: 0
(7999, 33)


In [8]:
# ------------------------------------------
# Handle Missing Values
# ------------------------------------------

clean_df["Zip Code - 3 digits"] = clean_df["Zip Code - 3 digits"].fillna("Unknown")

clean_df["CCSR Diagnosis Code"] = clean_df["CCSR Diagnosis Code"].fillna("Unknown")
clean_df["CCSR Diagnosis Description"] = clean_df["CCSR Diagnosis Description"].fillna("Unknown")

clean_df["CCSR Procedure Code"] = clean_df["CCSR Procedure Code"].fillna("No Procedure")
clean_df["CCSR Procedure Description"] = clean_df["CCSR Procedure Description"].fillna("No Procedure")

clean_df["APR Severity of Illness Description"] = (
    clean_df["APR Severity of Illness Description"].fillna("Unknown")
)

clean_df["APR Risk of Mortality"] = (
    clean_df["APR Risk of Mortality"].fillna("Unknown")
)

clean_df["Payment Typology 2"] = (
    clean_df["Payment Typology 2"].fillna("Not Applicable")
)

clean_df["Birth Weight"] = (
    clean_df["Birth Weight"].fillna("Not Applicable")
)

In [9]:
clean_df.drop(columns=["Payment Typology 3"], inplace=True)

In [10]:
print("Dataset Shape:")
print(clean_df.shape)

print("\nMissing Values After Cleaning:")
print(clean_df.isnull().sum())

Dataset Shape:
(7999, 32)

Missing Values After Cleaning:
Hospital Service Area                  0
Hospital County                        0
Operating Certificate Number           0
Permanent Facility Id                  0
Facility Name                          0
Age Group                              0
Zip Code - 3 digits                    0
Gender                                 0
Race                                   0
Ethnicity                              0
Length of Stay                         0
Type of Admission                      0
Patient Disposition                    0
Discharge Year                         0
CCSR Diagnosis Code                    0
CCSR Diagnosis Description             0
CCSR Procedure Code                    0
CCSR Procedure Description             0
APR DRG Code                           0
APR DRG Description                    0
APR MDC Code                           0
APR MDC Description                    0
APR Severity of Illness Code           0

In [11]:
# ------------------------------------------
# Check Unique Values in Categorical Columns
# ------------------------------------------

categorical_columns = [
    "Age Group",
    "Gender",
    "Race",
    "Ethnicity",
    "Type of Admission",
    "Patient Disposition",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "Payment Typology 1",
    "Payment Typology 2",
    "Emergency Department Indicator"
]

for column in categorical_columns:
    print("=" * 60)
    print(column)
    print("-" * 60)
    print(clean_df[column].value_counts())
    print()

Age Group
------------------------------------------------------------
Age Group
30 to 49       2043
0 to 17        1909
70 or Older    1850
50 to 69       1671
18 to 29        526
Name: count, dtype: int64

Gender
------------------------------------------------------------
Gender
F    4761
M    3238
Name: count, dtype: int64

Race
------------------------------------------------------------
Race
White                     3762
Other Race                2993
Black/African American     929
Multi-racial               315
Name: count, dtype: int64

Ethnicity
------------------------------------------------------------
Ethnicity
Not Span/Hispanic    5277
Unknown              1770
Spanish/Hispanic      952
Name: count, dtype: int64

Type of Admission
------------------------------------------------------------
Type of Admission
Emergency    4492
Elective     1723
Newborn      1321
Urgent        463
Name: count, dtype: int64

Patient Disposition
----------------------------------------------

In [12]:
# ------------------------------------------
# Validate Numeric Columns
# ------------------------------------------

numeric_columns = [
    "Length of Stay",
    "Total Charges",
    "Total Costs"
]

for column in numeric_columns:
    print("=" * 60)
    print(column)
    print("-" * 60)
    print("Minimum :", clean_df[column].min())
    print("Maximum :", clean_df[column].max())
    print("Mean    :", round(clean_df[column].mean(), 2))
    print("Median  :", round(clean_df[column].median(), 2))
    print()

Length of Stay
------------------------------------------------------------
Minimum : 1
Maximum : 120
Mean    : 5.73
Median  : 3.0

Total Charges
------------------------------------------------------------
Minimum : 7072.85
Maximum : 7411637.96
Mean    : 128062.14
Median  : 57694.25

Total Costs
------------------------------------------------------------
Minimum : 288.97
Maximum : 3129498.88
Mean    : 41199.91
Median  : 15698.75



In [13]:
print("Negative Length of Stay :", (clean_df["Length of Stay"] < 0).sum())
print("Negative Total Charges  :", (clean_df["Total Charges"] < 0).sum())
print("Negative Total Costs    :", (clean_df["Total Costs"] < 0).sum())

Negative Length of Stay : 0
Negative Total Charges  : 0
Negative Total Costs    : 0


In [14]:
# ------------------------------------------
# Save Cleaned Dataset
# ------------------------------------------

clean_df.to_csv(
    DATA_DIR / "hospital_cleaned.csv",
    index=False
)

print("hospital_cleaned.csv saved successfully.")

hospital_cleaned.csv saved successfully.


In [15]:
cleaned_df = pd.read_csv(DATA_DIR / "hospital_cleaned.csv")

print(cleaned_df.shape)
cleaned_df.head()

(7999, 32)


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,70 or Older,OOS,M,Other Race,Unknown,...,4,Extreme,Extreme,Medical,Medicare,Private Health Insurance,Not Applicable,N,121012.20,50785.63
1,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,0 to 17,114,F,Other Race,Spanish/Hispanic,...,2,Moderate,Minor,Medical,Medicaid,Not Applicable,Not Applicable,N,20798.65,9416.18
2,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,30 to 49,100,M,Black/African American,Spanish/Hispanic,...,2,Moderate,Moderate,Surgical,Medicaid,Not Applicable,Not Applicable,Y,124567.28,45270.48
3,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,70 or Older,103,F,Other Race,Unknown,...,3,Major,Major,Surgical,Medicare,Not Applicable,Not Applicable,N,325157.81,122411.36
4,New York City,Manhattan,7002054.0,1458.0,New York-Presbyterian Hospital - New York Weil...,0 to 17,100,M,White,Not Span/Hispanic,...,1,Minor,Minor,Medical,Private Health Insurance,Not Applicable,03500,N,21948.10,1071.12


## Final Result

Final Dataset Shape:
7999 rows × 32 columns

Duplicate Records:
0

Missing Values:
0 (for all retained columns)

Dataset Status:
Ready for KPI Engineering and Tableau Dashboard Development.